# Shapefile Generation Notebook

This notebook generates shapefiles bounding the data from specified regions within `amazon_data`.
It creates individual shapefiles for each source dataset folder and a combined shapefile.

In [ ]:
import os
import rasterio
from shapely.geometry import box
import geopandas as gpd
import pandas as pd

In [ ]:
# Configuration
base_data_dir = '/home/luizluz/Documentos/multi-task-fcn/amazon_data'
output_dir = '/home/luizluz/Documentos/multi-task-fcn/exploration_notebooks/shapes_output'
os.makedirs(output_dir, exist_ok=True)

# Define the regions (subdirectories in amazon_data)
# We assume each region contains training iterations with TIFF files.
# We will use the 'full_distance_map.tif' from 'iter_000/distance_map' or similar as the reference.
target_regions = ['13_amazon_data', '15_amazon_data']

# Function to find a representative TIFF in the region
def find_reference_tif(region_path):
    # Standard location based on file structure exploration
    candidate = os.path.join(region_path, 'iter_000', 'distance_map', 'full_distance_map.tif')
    if os.path.exists(candidate):
        return candidate
    # Fallback to search
    for root, dirs, files in os.walk(region_path):
        for f in files:
            if f.endswith('.tif'):
                return os.path.join(root, f)
    return None

In [ ]:
def get_bounds_df(region_name, tif_path):
    data = []
    try:
        with rasterio.open(tif_path) as src:
            bounds = src.bounds
            crs = src.crs
            geom = box(bounds.left, bounds.bottom, bounds.right, bounds.top)
            data.append({'region': region_name, 'path': tif_path, 'geometry': geom})
            return gpd.GeoDataFrame(data, crs=crs)
    except Exception as e:
        print(f"Error reading {tif_path}: {e}")
        return None

## Process Each Region

In [ ]:
gdfs = []

for region in target_regions:
    full_path = os.path.join(base_data_dir, region)
    print(f"Processing region: {region}...")
    
    ref_tif = find_reference_tif(full_path)
    if ref_tif:
        print(f"  Found reference TIFF: {ref_tif}")
        gdf = get_bounds_df(region, ref_tif)
        if gdf is not None:
            # Save individual shapefile
            shp_name = f"{region}_bounds.shp"
            shp_path = os.path.join(output_dir, shp_name)
            gdf.to_file(shp_path)
            print(f"  Saved shapefile to: {shp_path}")
            gdfs.append(gdf)
    else:
        print(f"  No suitable TIFF found in {full_path}")

## Merge Regions and Save Final Shapefile

In [ ]:
if gdfs:
    # Ensure CRS match (using the first one as reference)
    reference_crs = gdfs[0].crs
    aligned_gdfs = []
    
    for gdf in gdfs:
        if gdf.crs != reference_crs:
            print(f"Reprojecting {gdf.iloc[0]['region']} to match reference CRS...")
            aligned_gdfs.append(gdf.to_crs(reference_crs))
        else:
            aligned_gdfs.append(gdf)
            
    combined_gdf = pd.concat(aligned_gdfs, ignore_index=True)
    combined_shp_path = os.path.join(output_dir, 'all_regions_combined.shp')
    combined_gdf.to_file(combined_shp_path)
    print(f"Saved combined shapefile: {combined_shp_path}")
    print(combined_gdf)
else:
    print("No data to merge.")